# Python Refactorer – AI-Powered Code Modularizer

**Python Refactorer** is an interactive web application that leverages AI to automatically refactor messy, inefficient Python code into clean, modular, and maintainable code. Designed for developers, students, and engineers, it provides a professional-grade code improvement workflow directly in the browser.  

## Features

- **AI-Powered Refactoring:** Uses GPT-5 to analyze and restructure Python scripts while preserving functionality.
- **Code Modularization:** Breaks large scripts into reusable functions and classes following best practices.
- **Readability & Maintainability:** Improves naming, simplifies logic, and reduces code duplication.
- **Live Preview:** See the original and refactored code side by side.
- **Explanation Panel:** Understand the AI's reasoning behind each refactor.
- **Stream output on Gradio UI**


## Purpose

This project aims to accelerate Python code quality improvement, making scripts more professional, readable, and maintainable without manual intervention. It’s ideal for learning, prototyping, and production-ready code cleanup.

## Tech Stack

- **Frontend:** Gradio for interactive UI
- **Backend:** Python, OpenAI GPT-5 API
- **Styling:** Custom dark-mode CSS


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display
import gradio as gr

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
 

OpenAI API Key exists and begins sk-proj-


In [3]:
OPENAI_MODEL = "gpt-5"

In [ ]:
system_prompt = """ 
You are a senior Python software architect and expert code modularizer.
Your role is to transform messy, inefficient, poorly structured Python code into clean, modular, maintainable, and production-quality code while preserving the original functionality.
You think like an experienced engineer who cares deeply about software design, readability, maintainability, and performance.
Your primary objective is to refactor code into a well-organized structure using best practices from professional Python development.
CORE RESPONSIBILITIES
1. Code Refactoring
   Analyze the provided code and restructure it into modular, reusable, and logically separated components.
2. Modularization
   Break large functions or scripts into smaller focused functions or classes that follow the single-responsibility principle.
3. Improve Readability
   Ensure the code is easy to understand by:
* Using meaningful variable and function names
* Removing redundant logic
* Simplifying complex expressions
* Eliminating duplicated code
4. Improve Efficiency
   Optimize inefficient operations where appropriate without changing the program's behavior.
5. Apply Python Best Practices
   Follow modern Python conventions including:
* PEP 8 style guidelines
* Clear function signatures
* Proper use of classes where appropriate
* Logical file/module organization
* Consistent formatting
6. Improve Maintainability
   Ensure the refactored code:
* Is easy to extend
* Has clear boundaries between components
* Separates concerns properly
* Avoids tightly coupled logic
7. Add Structure
   Where useful, introduce:
* helper functions
* utility modules
* configuration constants
* reusable components
* clear entry points (e.g., main functions)
8. Documentation
   Provide concise docstrings for modules, classes, and functions explaining their purpose and usage.
9. Error Handling
   Add safe and appropriate error handling where necessary without overcomplicating the code.
10. Preserve Behavior
    The refactored code must produce the same results as the original code
    
OUTPUT FORMAT
When responding:
1. Briefly describe the main problems in the original code.
2. Explain the modularization strategy used.
3. Provide the fully refactored Python code.
4. Ensure the code is clean, properly formatted, and easy to copy and run

REFACTORING PRINCIPLES
Follow these engineering principles:
* Single Responsibility Principle
* Separation of Concerns
* DRY (Don't Repeat Yourself)
* Clear Abstractions
* Minimal Side Effects
* Logical Function Decomposition
* Readable Control Flow

CODE QUALITY EXPECTATIONS
Your refactored code should:
* Be significantly easier to read
* Reduce complexity
* Use clear function boundaries
* Avoid large monolithic functions
* Use descriptive naming
* Follow Pythonic idioms

DO NOT

* Change the intended functionality
* Introduce unnecessary complexity
* Add excessive comments that repeat obvious code behavior
* Over-engineer simple logic

THINK LIKE A PROFESSIONAL SOFTWARE ENGINEER

Always prioritize clarity, structure, and maintainability. The goal is to transform chaotic scripts into clean, professional Python code that could belong in a well-maintained production repository.
Before writing the final refactored code, you must carefully analyze the
original code and explain your reasoning.

"""

def user_prompt(python_code):
    return f"""
 
    Refactor the following Python code to make it clean, modular, and efficient.

Goals:

Break the code into smaller reusable functions

Improve readability and naming

Remove duplicated logic

Follow Python best practices (PEP 8)

Add useful docstrings

Preserve the original functionality

Explain briefly what problems exist in the original code and how you improved it.

Then provide the fully refactored code.

Here is the messy code: 
```python
{python_code}
```
    """

In [5]:
def message_builder(python_code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt(python_code)}
    ]

In [6]:
openai = OpenAI()

In [ ]:
import re

def refactor_code_stream(model_name, python_code):
    messages = message_builder(python_code)
    response = ""
    code_buffer = ""

    stream = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages,
        stream=True
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        delta_content = delta.content
        if delta_content is not None:
            response += delta_content

            code_match = re.search(r"```python(.*?)```", response, re.DOTALL)
            if code_match:
                code_buffer = code_match.group(1)
                explanation = response.replace(code_match.group(0), "").strip()
            else:
                code_buffer = response
                explanation = ""

            yield code_buffer, explanation

    code_match = re.search(r"```python(.*?)```", response, re.DOTALL)
    if code_match:
        code_buffer = code_match.group(1)
        explanation = response.replace(code_match.group(0), "").strip()
    else:
        code_buffer = response
        explanation = ""

    yield code_buffer, explanation  
          

In [ ]:
import gradio as gr
import re


def run_python(code):
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return str(e)


with open("styles.css", "r") as f:
    css_content = f.read() 
with gr.Blocks(css=css_content, theme=gr.themes.Monochrome(), title="Python Refactorer") as ui:

  
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_input = gr.Code(
                label="Original Python Code",
                language="python",
                lines=10,
            )
        with gr.Column(scale=6):
            refactored_output = gr.Code(
                label="Refactored Python Code",
                language="python",
                lines=10,
                show_line_numbers=True,
                interactive=False
            )


    with gr.Row(elem_classes=["controls"]):
        model_dropdown = gr.Dropdown(["gpt-5"], value="gpt-5", show_label=False)
        refactor_btn = gr.Button("Refactor Code", elem_classes=["convert-btn"])

 
    explanation_panel = gr.Markdown(label="Refactoring Explanation", elem_classes=["explanation"])



    refactor_btn.click(
        fn=refactor_code_stream,
        inputs=[model_dropdown, python_input],
        outputs=[refactored_output, explanation_panel],
        queue=True
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.
